# Análise dos resultados — Trabalho 4 (Link Extractor)

Este notebook consolida os CSVs gerados pelo Locust em `results/raw/`,
produz a planilha `results/processed/consolidado.xlsx` e gera os gráficos
comparativos em `results/charts/`.

**Pré-requisito**: ter rodado `scripts/run-all-scenarios.ps1` (ou cenários
individuais) para popular `results/raw/`.

**Como executar**:
```powershell
.\.venv\Scripts\jupyter.exe lab analysis/analise.ipynb
```
ou abra em VSCode com a extensão Jupyter.

## 1. Imports e caminhos

In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
RAW = ROOT / "results" / "raw"
PROCESSED = ROOT / "results" / "processed"
CHARTS = ROOT / "results" / "charts"
PROCESSED.mkdir(parents=True, exist_ok=True)
CHARTS.mkdir(parents=True, exist_ok=True)

print(f"ROOT      = {ROOT}")
print(f"RAW       = {RAW}")
print(f"PROCESSED = {PROCESSED}")
print(f"CHARTS    = {CHARTS}")
print(f"\nCSVs encontrados em raw: {len(list(RAW.glob('*_stats.csv')))}")

## 2. Consolidação dos CSVs do Locust

Cada cenário gera 4 arquivos com prefixo `<version>_<cache>_<users>u_<runtime>`.
Vamos extrair a linha `Aggregated` de cada `*_stats.csv` para montar a aba
`resumo`, e juntar todos os `*_stats_history.csv` na aba `series`.

In [ ]:
NAME_RE = re.compile(
    r"^(?P<version>python|ruby)_(?P<cache>warm|cold)_(?P<users>\d+)u_(?P<runtime>[^_]+)_stats\.csv$"
)

def parse_tag(filename: str):
    m = NAME_RE.match(filename)
    if not m:
        return None
    return {
        "version": m.group("version"),
        "cache": m.group("cache"),
        "users": int(m.group("users")),
        "runtime": m.group("runtime"),
    }

rows = []
for csv in sorted(RAW.glob("*_stats.csv")):
    tag = parse_tag(csv.name)
    if tag is None:
        continue
    df = pd.read_csv(csv)
    agg = df[df["Name"] == "Aggregated"]
    if agg.empty:
        agg = df.tail(1)
    r = agg.iloc[0].to_dict()
    rows.append({
        **tag,
        "requests": int(r.get("Request Count", 0)),
        "failures": int(r.get("Failure Count", 0)),
        "rps": r.get("Requests/s", float("nan")),
        "mean_ms": r.get("Average Response Time", float("nan")),
        "median_ms": r.get("Median Response Time", float("nan")),
        "p95_ms": r.get("95%", float("nan")),
        "p99_ms": r.get("99%", float("nan")),
        "min_ms": r.get("Min Response Time", float("nan")),
        "max_ms": r.get("Max Response Time", float("nan")),
    })

if not rows:
    raise RuntimeError("Nenhum *_stats.csv em results/raw/. Rode os cenários antes.")

resumo = pd.DataFrame(rows).sort_values(["version", "cache", "users"]).reset_index(drop=True)
resumo

In [ ]:
frames = []
for csv in sorted(RAW.glob("*_stats_history.csv")):
    base = csv.name.replace("_stats_history.csv", "_stats.csv")
    tag = parse_tag(base)
    if tag is None:
        continue
    df = pd.read_csv(csv)
    for k, v in tag.items():
        df[k] = v
    frames.append(df)

series = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print(f"Linhas na série temporal consolidada: {len(series)}")
series.head()

## 3. Salvar a planilha consolidada

In [ ]:
out_path = PROCESSED / "consolidado.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as w:
    resumo.to_excel(w, sheet_name="resumo", index=False)
    if not series.empty:
        series.to_excel(w, sheet_name="series", index=False)

print(f"Salvo: {out_path}")

## 4. Gráficos

Cada gráfico compara as quatro combinações `versao/cache` ao longo dos
níveis de carga (10, 50, 100 usuários).

In [ ]:
def line_by_users(df, metric, ylabel, title, fname):
    fig, ax = plt.subplots(figsize=(8, 5))
    for (version, cache), grp in df.groupby(["version", "cache"]):
        grp = grp.sort_values("users")
        ax.plot(grp["users"], grp[metric], marker="o", label=f"{version} / {cache}")
    ax.set_xlabel("Usuários virtuais")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    out = CHARTS / fname
    fig.savefig(out, dpi=120)
    plt.show()
    print(f"  salvo: {out}")

### 4.1. Tempo médio de resposta

In [ ]:
line_by_users(resumo, "mean_ms", "Tempo médio (ms)",
              "Tempo médio de resposta por carga",
              "tempo_resposta_por_carga.png")

### 4.2. Mediana

In [ ]:
line_by_users(resumo, "median_ms", "Mediana (ms)",
              "Mediana do tempo de resposta por carga",
              "mediana_por_carga.png")

### 4.3. Percentil 95

In [ ]:
line_by_users(resumo, "p95_ms", "p95 (ms)",
              "Percentil 95 do tempo de resposta por carga",
              "p95_por_carga.png")

### 4.4. Percentil 99

In [ ]:
line_by_users(resumo, "p99_ms", "p99 (ms)",
              "Percentil 99 do tempo de resposta por carga",
              "p99_por_carga.png")

### 4.5. Throughput (RPS)

In [ ]:
line_by_users(resumo, "rps", "Requisições/s",
              "Throughput por carga",
              "throughput_por_carga.png")

### 4.6. Falhas por cenário

In [ ]:
df = resumo.copy()
df["scenario"] = df["version"] + "/" + df["cache"] + "/" + df["users"].astype(str) + "u"
df = df.sort_values(["version", "cache", "users"])

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(df["scenario"], df["failures"])
ax.set_xlabel("Cenário")
ax.set_ylabel("Falhas (HTTP não-200)")
ax.set_title("Falhas por cenário")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
out = CHARTS / "falhas_por_cenario.png"
fig.savefig(out, dpi=120)
plt.show()
print(f"  salvo: {out}")

## 5. Observações finais

Use esta seção para anotar conclusões durante a análise:

- Qual versão (Python ou Ruby) tem **menor latência média** e por que.
- Quão grande é o ganho do cache (warm vs cold) em cada versão.
- A partir de qual nível de carga começam falhas e em qual combinação.
- Diferença entre **mediana e p99** — se p99 cresce muito mais rápido,
  há cauda longa (alguns clientes esperam muito mais que a maioria).
- Como o **throughput satura** com o aumento de usuários — se a curva
  achata, atingimos o limite do serviço.